In [ ]:
import pandas as pd
import sklearn.model_selection

# 1. โหลดข้อมูล
df = pd.read_csv("../raw_data/bank-full.csv", sep=";")

# 2. ลบข้อมูลที่ซ้ำกัน (Drop Duplicates)
df = df.drop_duplicates()

# 3. ลบคอลัมน์ duration เพื่อป้องกัน Data Leakage
df = df.drop(columns=['duration'])

# 4. จัดการ pdays และสร้าง previously_contacted
df["previously_contacted"] = (df["pdays"] != -1).astype(int)
df["pdays"] = df["pdays"].replace(-1, 0)

# 5. ลบแถวที่มี unknown ใน job และ education
df = df[(df['job'] != 'unknown') & (df['education'] != 'unknown')].copy()

# 6. จัดการ unknown และ other ในกลุ่มหมวดหมู่
df['contact'] = df['contact'].replace('unknown', 'unrecorded')
df['poutcome'] = df['poutcome'].replace(['unknown', 'other'], 'nonexistent')

# 7. แปลง Target 'y' เป็นตัวเลข
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# 8. แบ่งข้อมูล Train/Test (เพิ่ม stratify เพื่อรักษาสัดส่วนของ y ให้เท่ากัน)
train_df, test_df = sklearn.model_selection.train_test_split(
    df, train_size=0.80, shuffle=True, random_state=0, stratify=df['y']
)

# 9. บันทึกไฟล์
train_df.to_csv("../cleaned_data/train_data.csv", index=False)
test_df.to_csv("../cleaned_data/test_data.csv", index=False)

print("Data Cleansing & Split Completed!")
print(f"Train Shape: {train_df.shape}, Test Shape: {test_df.shape}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from IPython.display import display, HTML


# ==========================================
# 1. โหลดและจัดการข้อมูล
# ==========================================
raw_df = pd.read_csv("../raw_data/bank-full.csv", sep=";")

df = raw_df.copy()
df = df.drop_duplicates()
df = df.drop(columns=['duration'])
df["previously_contacted"] = (df["pdays"] != -1).astype(int)
df["pdays"] = df["pdays"].replace(-1, 0)
df = df[(df['job'] != 'unknown') & (df['education'] != 'unknown')].copy()
df['contact'] = df['contact'].replace('unknown', 'unrecorded')
df['poutcome'] = df['poutcome'].replace(['unknown', 'other'], 'nonexistent')
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# ==========================================
# 2. สร้างกราฟพร้อมตัวเลขกำกับทุกแท่ง
# ==========================================

print(" BEFORE CLEANSING (Raw Data - 10 Rows):")
display(raw_df.head(10))

print("\n AFTER CLEANSING (Cleaned Data - 10 Rows):")
display(df.head(10))

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.subplots_adjust(hspace=0.5, wspace=0.3)

# --- กราฟ 1: จำนวนแถวทั้งหมด ---
stages = ['Before (Raw Data)', 'After (Cleaned Data)']
counts = [len(raw_df), len(df)]
bars = axes[0, 0].bar(stages, counts, color=['#e74c3c', '#2ecc71'], width=0.5)
axes[0, 0].set_title("1. Total Row Count Reduction", fontweight='bold')
axes[0, 0].set_ylabel("Number of Rows")
axes[0, 0].set_ylim(0, max(counts) * 1.15)
for bar in bars:
    yval = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, yval + 500, f"{yval:,}", ha='center', fontweight='bold')

# --- กราฟ 2: การกระจายตัวของ pdays (แสดงให้เห็นความต่าง) ---
# ใช้ bar plot แทน histogram เพื่อให้ใส่ตัวเลขบนแท่งเฉพาะกลุ่มค่าที่สำคัญได้ง่าย
pdays_before_counts = [len(raw_df[raw_df['pdays'] == -1]), len(raw_df[raw_df['pdays'] > -1])]
pdays_after_counts = [len(df[df['pdays'] == 0]), len(df[df['pdays'] > 0])]

axes[0, 1].bar(['-1 (Not Contacted)', 'Other (> -1)'], pdays_before_counts, alpha=0.6, color='#e74c3c', label='Before')
axes[0, 1].bar(['0 (Replaced -1)', 'Other (> 0)'], pdays_after_counts, alpha=0.6, color='#2ecc71', label='After')
axes[0, 1].set_title("2. 'pdays' Transformation", fontweight='bold')
axes[0, 1].legend()
for container in axes[0, 1].containers:
    axes[0, 1].bar_label(container, fmt='%d', padding=3, fontweight='bold')

# --- กราฟ 3: หมวดหมู่ในคอลัมน์ contact ---
sns.countplot(data=raw_df, x='contact', ax=axes[1, 0], palette='Reds', order=raw_df['contact'].value_counts().index)
axes[1, 0].set_title("3A. 'contact' BEFORE ('unknown' exists)", fontweight='bold')
for container in axes[1, 0].containers:
    axes[1, 0].bar_label(container, fmt='%d', padding=3, fontweight='bold')

sns.countplot(data=df, x='contact', ax=axes[1, 1], palette='Greens', order=df['contact'].value_counts().index)
axes[1, 1].set_title("3B. 'contact' AFTER ('unknown' -> 'unrecorded')", fontweight='bold')
for container in axes[1, 1].containers:
    axes[1, 1].bar_label(container, fmt='%d', padding=3, fontweight='bold')

# --- กราฟ 4: หมวดหมู่ในคอลัมน์ poutcome ---
sns.countplot(data=raw_df, x='poutcome', ax=axes[2, 0], palette='Reds', order=raw_df['poutcome'].value_counts().index)
axes[2, 0].set_title("4A. 'poutcome' BEFORE ('unknown', 'other' exist)", fontweight='bold')
for container in axes[2, 0].containers:
    axes[2, 0].bar_label(container, fmt='%d', padding=3, fontweight='bold')

sns.countplot(data=df, x='poutcome', ax=axes[2, 1], palette='Greens', order=df['poutcome'].value_counts().index)
axes[2, 1].set_title("4B. 'poutcome' AFTER (merged into 'nonexistent')", fontweight='bold')
for container in axes[2, 1].containers:
    axes[2, 1].bar_label(container, fmt='%d', padding=3, fontweight='bold')

plt.show()